In [2]:
import numpy               as np
#import seaborn             as sns
import pandas              as pd
import libraries.plotting  as slp
import libraries.utilities as sul
import os

#sns.set_theme()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We only consider coherent interfaces (which has matching lattices from both sides of the interface, meaning that there is a repeat along the interface that lets build periodicity along the interface surface).

In [13]:
# Define name to reference folder which stores the energies of each slab
slab_folder = f'/Users/cibran/Desktop/MChX-slabs/MChX/0.0_0.0_0.0/slabs'

data = {}
for slab in os.listdir('/Users/cibran/Desktop/MChX-slabs/MChX'):
    slab_folder = f'/Users/cibran/Desktop/MChX-slabs/MChX/{slab}/slabs'
    try:
        bulk_folder = f'{slab_folder}/bulk'
        
        # Load the bulk energy
        bulk_energy = sul.read_energy(bulk_folder)
        
        # Load bulka information
        slab_data = sul.load_json(filename=f'{bulk_folder}/slab_data.json')
        
        n_bulk = slab_data['number_of_sites']
        
        # Initialize the data dictionary for storing all slab energy calculations
        # Iterate over slabs
        surface_energies_of_formation = {}
        for i, miller_index_str in enumerate(os.listdir(slab_folder)):
            # Define current folder
            miller_folder = f'{slab_folder}/{miller_index_str}'
            print(miller_folder)
            # Skip in case it is not a folder or it is the bulk folder
            if (not os.path.isdir(miller_folder)) or (miller_index_str == 'bulk'):
                continue
        
            # Load slab information
            slab_data = sul.load_json(filename=f'{miller_folder}/slab_data.json')
        
            print()
            print(f'Slab {i+1}')
            print()
            print(f'Miller index: {slab_data['miller_index']}')
            print(f'Shift: {slab_data['shift']:.3g}')
            print(f'Surface area: {slab_data['surface_area']:.3g}')
            print(f'Number of sites: {slab_data['number_of_sites']}')
        
            # Load the single-shot energy
            slab_energy = sul.read_energy(miller_folder)
        
            print(f'E_total = {slab_energy:.3g} eV')
        
            bulk_energy_times_fu = bulk_energy * slab_data['number_of_sites'] / n_bulk
            
            # Compute surface energy of formation in eV/atom/ang^2
            surface_energy_of_formation = sul.get_surface_energy_of_formation(slab_energy, bulk_energy_times_fu, slab_data['surface_area'])
        
            # Save the slab energy
            np.savetxt(f'{miller_folder}/surface_energy_of_formation', [surface_energy_of_formation])
        
            print(f'\t$E_surface$ = {surface_energy_of_formation:.3g} J/m²')
        
            # Generate a dictionary object with the new data and update in the main data object
            surface_energies_of_formation.update({
                miller_index_str: surface_energy_of_formation
            })
        
        # Convert to Pandas DataFrame
        surface_energies_of_formation = pd.DataFrame(surface_energies_of_formation, index=['energy'])
        
        min_arg      = np.argsort(surface_energies_of_formation.values)[0]
        local_minima = surface_energies_of_formation.columns[min_arg]
        energies     = surface_energies_of_formation.values[0][min_arg]
        local_minima, energies
        
        print('0_1_0 - 0_1_1')
        if local_minima[0][:5] == '0_1_1':
            diff = energies[1] - energies[0]
        elif local_minima[0][:5] == '0_1_0':
            diff = energies[0] - energies[1]
        else:
            print('HEEY')
        print(diff)
        
        data.update({
            slab: diff
        })
    except:
        print(f'No {slab}')
        pass

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pymatgen/io/vasp/inputs.py:2750: UnknownPotcarWarning: POTCAR data with symbol Sb is not known to pymatgen. Your POTCAR may be corrupted or pymatgen's POTCAR database is incomplete.
  psingle = PotcarSingle(f"{p_strip}\nEnd of Dataset\n")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pymatgen/io/vasp/inputs.py:2750: UnknownPotcarWarning: POTCAR data with symbol I is not known to pymatgen. Your POTCAR may be corrupted or pymatgen's POTCAR database is incomplete.
  psingle = PotcarSingle(f"{p_strip}\nEnd of Dataset\n")


/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.0_0.75/slabs/INCAR
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.0_0.75/slabs/bulk
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.0_0.75/slabs/0_1_0_i_23

Slab 3

Miller index: [0, 1, 0]
Shift: 0.25
Surface area: 44
Number of sites: 72
E_total = -130 eV
	$E_surface$ = 0.118 J/m²
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.0_0.75/slabs/.DS_Store
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.0_0.75/slabs/POSCAR
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.0_0.75/slabs/0_1_1_i_19

Slab 6

Miller index: [0, 1, 1]
Shift: 0.5
Surface area: 56.3
Number of sites: 96
E_total = -173 eV
	$E_surface$ = 0.102 J/m²
0_1_0 - 0_1_1
0.01600019720500906
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.25_0.5/slabs/INCAR
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.25_0.5/slabs/bulk
/Users/cibran/Desktop/MChX-slabs/MChX/0.75_0.25_0.5/slabs/0_1_0_i_23

Slab 3

Miller index: [0, 1, 0]
Shift: 0.25
Surface area: 42.9
Number of sites: 72
E_total = -125 eV
	$E_surface$ = 0.

In [16]:
data

{'0.75_0.0_0.75': np.float64(0.01600019720500906),
 '0.75_0.25_0.5': np.float64(0.016988220582603436),
 '0.25_0.0_1.0': np.float64(0.013260695358369781),
 '1.0_0.5_1.0': np.float64(0.01872039911035442),
 '0.5_0.25_0.5': np.float64(0.016917242289083173),
 '0.5_0.0_0.75': np.float64(0.015879807607467378),
 '0.25_1.0_0.5': np.float64(0.018390462125566803),
 '0.75_0.0_0.0': np.float64(0.011754023672401706),
 '0.5_1.0_0.25': np.float64(0.018270488015629027),
 '0.25_0.25_0.25': np.float64(0.01493091175010304),
 '0.0_0.5_0.5': np.float64(0.01600640491419944),
 '0.5_1.0_0.0': np.float64(0.01728032868280216),
 '1.0_0.25_0.25': np.float64(0.0153208886250122),
 '0.75_1.0_0.25': np.float64(0.020122005033980414),
 '0.25_0.75_1.0': np.float64(0.017607738715686286),
 '0.0_0.25_0.75': np.float64(0.015692222810527387),
 '0.75_0.5_0.0': np.float64(0.015038664725574746),
 '0.0_0.75_1.0': np.float64(0.016081476228090696),
 '0.0_1.0_1.0': np.float64(0.016260712444801276),
 '1.0_0.25_1.0': np.float64(0.0186

In [20]:
# Maximum
max_key = max(data, key=data.get)
print("Max:", max_key, data[max_key])

# Minimum
min_key = min(data, key=data.get)
print("Min:", min_key, data[min_key])

Max: 1.0_1.0_0.75 0.023858358898610155
Min: 0.75_0.0_0.0 0.011754023672401706
